In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib, json, shutil, zipfile, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.signal import butter, sosfiltfilt
from scipy.stats import spearmanr
try:
    from IPython.display import display, HTML
except ImportError:
    display = print
    HTML = lambda text: text

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/hitting'
OUT = ROOT / 'results/pelvis_torso_uprighting_offset_v3'
FIG = ROOT / 'figures/pelvis_torso_uprighting_offset_v3'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)

AUTO_DOWNLOAD = True
MAKE_ANIMATION = True
SELECTED_SWING = None
LOOKBACK_S = .150
RATE_FLOOR_DPS = 20.
PEAK_FRACTION = .15
SUSTAIN_S = .010
FILTER_HZ = None
MIN_VALID_SWINGS = 5
REFERENCE_PLANT_CONTACT_RANGE_MS = (80., 200.)
LAG_INSPECTION_THRESHOLD_MS = 100.
MAX_EXTREME_TRIAL_PLOTS = 4
FINAL_EPISODE_CONTACT_GAP_S = .030
LOADING_TO_PLANT_INSPECTION_MAX_MS = 150.
BOOTSTRAPS = 1000
SEED = 20261009
EVENTS = ['fp_10_time', 'fp_100_time', 'contact_time']
SEGMENTS = ('pelvis', 'torso')
ANGLE_COLS = [s + '_angle_x' for s in SEGMENTS]
OUTCOMES = {'bat_speed_mph': 'bat_speed_mph_contact_x',
            'exit_velocity_mph': 'exit_velo_mph_x',
            'attack_angle_deg': 'attack_angle_contact_x'}
ARCHIVE_HASHES = {
    'hitting_joint_angles.zip': '98a405c5c4a27c1a599d3e30b39a3087180cc4d4bb69faf62204a1c06881be95',
    'hitting_landmarks.zip': '9cc2bdc94f5f5a6c5d540f4abe459caae2c5164cd53f6a6fd23f4b2b7283bcb3'}
BASE = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/'
CSV_BASE = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_hitting/data/'
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'pelvis': '#167D9A', 'torso': '#A83C51'}
print('Working directory:', ROOT)
print('Signed lag: torso onset minus pelvis onset; positive = pelvis first.')

print("Duration restriction is a sensitivity cohort, not an exclusion from the primary analysis.")
print("Large-lag flags are inspection aids; they do not identify bad data automatically.")

print('Final episode: last contiguous positive-rate episode with a sustained threshold crossing,')
print('ending within 30 ms of contact. No fallback to an earlier episode outside that gap.')
print('Thresholds match the original detector; zero crossings separate movement episodes.')
print('Episode proximity and loading-duration flags are analytical choices, not clinical cutoffs.')

In [ ]:
def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, target):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {target}. Supply official data or set AUTO_DOWNLOAD=True.')
    print('Downloading', target.name, flush=True)
    temporary = target.with_suffix(target.suffix + '.part')
    urlretrieve(url, temporary)
    temporary.replace(target)

def ensure_archive(archive_name, csv_name):
    target = RAW / csv_name
    if target.exists():
        return target
    archive = RAW / archive_name
    if not archive.exists():
        download(BASE + archive_name, archive)
    if file_hash(archive) != ARCHIVE_HASHES[archive_name]:
        raise ValueError(f'Checksum mismatch: {archive.name}; remove the archive and retry.')
    with zipfile.ZipFile(archive) as z:
        members = [name for name in z.namelist() if Path(name).name == csv_name]
        if len(members) != 1:
            raise ValueError(f'Expected exactly one {csv_name} in {archive_name}')
        temporary = target.with_suffix('.csv.part')
        with z.open(members[0]) as source, temporary.open('wb') as dest:
            shutil.copyfileobj(source, dest)
        temporary.replace(target)
    return target

ensure_archive('hitting_joint_angles.zip', 'joint_angles.csv')
if not (RAW / 'metadata.csv').exists():
    download(CSV_BASE + 'metadata.csv', RAW / 'metadata.csv')
angles = pd.read_csv(RAW / 'joint_angles.csv',
                     usecols=['session_swing', 'time'] + EVENTS + ANGLE_COLS,
                     dtype={'session_swing': str})
metadata = pd.read_csv(RAW / 'metadata.csv',
                       dtype={'session_swing': str, 'user': str, 'session': str})
required_metadata = ['session_swing', 'user', 'session', 'hitter_side']
if not set(required_metadata).issubset(metadata.columns):
    raise ValueError('Missing required metadata columns')
if metadata.session_swing.isna().any() or metadata.session_swing.duplicated().any():
    raise ValueError('Missing or duplicate metadata swing IDs')
if angles.session_swing.isna().any() or angles.duplicated(['session_swing', 'time']).any():
    raise ValueError('Missing swing IDs or duplicate angle swing/time keys')
meta = metadata.set_index('session_swing')
groups = {str(k): g.sort_values('time').reset_index(drop=True)
          for k, g in angles.groupby('session_swing', sort=False)}

poi = pd.DataFrame(columns=['session_swing'] + list(OUTCOMES.values()))
try:
    if not (RAW / 'poi_metrics.csv').exists():
        download(CSV_BASE + 'poi/poi_metrics.csv', RAW / 'poi_metrics.csv')
    poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing': str})
except (OSError, FileNotFoundError) as exc:
    print('Optional outcomes unavailable; timing analysis continues:', exc)
if poi.session_swing.isna().any() or poi.session_swing.duplicated().any():
    raise ValueError('Missing or duplicate POI swing IDs')
for column in OUTCOMES.values():
    if column not in poi:
        poi[column] = np.nan
print(f'{len(groups):,} angle trials; {metadata.user.nunique():,} hitters in metadata')

In [ ]:
def event_values(g):
    result = {}
    for name in EVENTS:
        values = g[name].to_numpy(float)
        if not np.isfinite(values).all() or np.ptp(values) > 1e-7:
            raise ValueError('missing or inconsistent event annotation')
        result[name] = float(values[0])
    if not result['fp_10_time'] < result['fp_100_time'] < result['contact_time']:
        raise ValueError('unordered official events')
    return result

def prepared_trace(g, lookback=LOOKBACK_S, cutoff=FILTER_HZ):
    events = event_values(g)
    lo = events['fp_10_time'] - lookback
    hi = events['contact_time']
    pad = .100 if cutoff is not None else .025
    t_all = g.time.to_numpy(float)
    if not np.isfinite(t_all).all() or np.any(np.diff(t_all) <= 0):
        raise ValueError('invalid time grid')
    if t_all[0] > lo - pad or t_all[-1] < hi + pad:
        raise ValueError('insufficient window/context coverage')
    left = max(0, np.searchsorted(t_all, lo-pad, side='right')-1)
    right = min(len(t_all), np.searchsorted(t_all, hi+pad, side='left')+1)
    t = t_all[left:right]
    dt = float(np.median(np.diff(t)))
    if len(t) < 8 or not np.allclose(np.diff(t), dt, rtol=.03, atol=.00011):
        raise ValueError('sampling gap or irregular grid')
    trace = {'time': t, 'events': events, 'lo': lo, 'hi': hi, 'dt': dt,
             'row_indices': np.arange(left, right),
             'mask': (t >= lo - 1e-10) & (t <= hi + 1e-10)}
    if trace['mask'].sum() < 8:
        raise ValueError('too few search samples')
    for s in SEGMENTS:
        a = g[s+'_angle_x'].to_numpy(float)[left:right]
        if not np.isfinite(a).all():
            raise ValueError('nonfinite angle in analysis/context window')
        a = np.rad2deg(np.unwrap(np.deg2rad(a)))
        if cutoff is not None:
            if not 0 < cutoff < .5/dt:
                raise ValueError('invalid filter cutoff')
            a = sosfiltfilt(butter(4, cutoff, fs=1/dt, output='sos'), a)
        trace[s+'_angle'] = a
        trace[s+'_rate'] = np.gradient(a, t)
    return trace

def detect_onset(time, rate, floor=RATE_FLOOR_DPS,
                 fraction=PEAK_FRACTION, sustain=SUSTAIN_S):
    t, r = np.asarray(time, float), np.asarray(rate, float)
    if len(t) < 2 or not np.isfinite(t).all() or not np.isfinite(r).all():
        raise ValueError('invalid onset inputs')
    if len(t) != len(r) or np.any(np.diff(t) <= 0):
        raise ValueError('invalid onset time grid')
    if floor <= 0 or not 0 <= fraction <= 1 or sustain <= 0:
        raise ValueError('invalid onset settings')
    threshold = max(float(floor), fraction * max(0., float(r.max())))
    above = r >= threshold
    starts = np.flatnonzero(above & ~np.r_[False, above[:-1]])
    ends = np.flatnonzero(above & ~np.r_[above[1:], False])
    for start, end in zip(starts, ends):
        if t[end] - t[start] + 1e-10 >= sustain:
            state = 'left_censored' if start == 0 else 'detected'
            return {'state': state, 'time': float(t[start]) if start else np.nan,
                    'sample_index': int(start) if start else np.nan,
                    'threshold_dps': threshold}
    return {'state': 'not_detected', 'time': np.nan, 'sample_index': np.nan,
            'threshold_dps': threshold}

def extract(g, lookback=LOOKBACK_S, cutoff=FILTER_HZ,
            floor=RATE_FLOOR_DPS, fraction=PEAK_FRACTION, sustain=SUSTAIN_S):
    tr = prepared_trace(g, lookback, cutoff)
    m, t, e = tr['mask'], tr['time'], tr['events']
    search_t = t[m]
    amount_t = np.r_[tr['lo'], t[(t > tr['lo']) & (t < tr['hi'])], tr['hi']]
    row = {**e, 'sample_interval_ms': 1000*tr['dt'], 'search_start_s': tr['lo'],
           'plant_to_contact_ms': 1000*(e['contact_time']-e['fp_100_time']),
           'loading_to_plant_ms': 1000*(e['fp_100_time']-e['fp_10_time']),
           'search_duration_ms': 1000*(tr['hi']-tr['lo'])}
    for s in SEGMENTS:
        rates = tr[s+'_rate'][m]
        onset = detect_onset(search_t, rates, floor, fraction, sustain)
        on_t = onset['time']
        a = np.interp(amount_t, t, tr[s+'_angle'])
        row.update({s+'_onset_state': onset['state'],
                    s+'_threshold_dps': onset['threshold_dps'],
                    s+'_onset_s': on_t,
                    s+'_onset_ms_from_plant': 1000*(on_t-e['fp_100_time']),
                    s+'_onset_ms_from_foot_contact': 1000*(on_t-e['fp_10_time']),
                    s+'_onset_sample_row': (int(tr['row_indices'][m][int(onset['sample_index'])])
                                           if onset['state']=='detected' else np.nan),
                    s+'_peak_rate_dps': float(rates.max()),
                    s+'_net_change_deg': float(a[-1]-a[0]),
                    s+'_max_positive_excursion_deg': float(max(0., (a-a[0]).max()))})
    both = all(row[s+'_onset_state']=='detected' for s in SEGMENTS)
    lag = 1000*(row['torso_onset_s']-row['pelvis_onset_s']) if both else np.nan
    row['uprighting_offset_ms'] = lag
    row['uprighting_offset_samples'] = lag/(1000*tr['dt']) if both else np.nan
    row['onset_pair_valid'] = both
    row['onset_order'] = ('unavailable' if not both else
                          'unresolved_within_one_sample' if abs(lag)<=1000*tr['dt']+1e-7 else
                          'pelvis_first' if lag>0 else 'torso_first')
    row['torso_minus_pelvis_peak_rate_dps'] = row['torso_peak_rate_dps']-row['pelvis_peak_rate_dps']
    return row

def build_table(**settings):
    rows, audit = [], []
    for key, g in groups.items():
        try:
            if key not in meta.index:
                raise ValueError('missing metadata match')
            identity = meta.loc[key]
            if identity[['user', 'session']].isna().any() or identity.hitter_side not in ('L','R'):
                raise ValueError('missing identity or handedness')
            row = extract(g, **settings)
            rows.append({'session_swing': key, 'hitter_id': str(identity.user),
                         'session_id': str(identity.session), 'hitter_side': identity.hitter_side, **row})
            status = 'retained'
        except (ValueError, KeyError, TypeError) as exc:
            status = str(exc)
        audit.append({'session_swing': key, 'status': status})
    return pd.DataFrame(rows), pd.DataFrame(audit)


def sustained_runs(t,above,sustain):
    starts = np.flatnonzero(above & ~np.r_[False,above[:-1]])
    ends = np.flatnonzero(above & ~np.r_[above[1:],False])
    return [(int(a),int(b)) for a,b in zip(starts,ends) if t[b]-t[a]+1e-10>=sustain]

def detect_final_episode(time,rate,contact,floor=RATE_FLOOR_DPS,
                         fraction=PEAK_FRACTION,sustain=SUSTAIN_S,
                         contact_gap=FINAL_EPISODE_CONTACT_GAP_S):
    t,r = np.asarray(time,float),np.asarray(rate,float)
    if len(t)<2 or len(t)!=len(r) or not np.isfinite(t).all() or not np.isfinite(r).all():
        raise ValueError('invalid final-episode inputs')
    if np.any(np.diff(t)<=0) or floor<=0 or not 0<=fraction<=1 or sustain<=0 or contact_gap<0:
        raise ValueError('invalid final-episode settings')
    threshold = max(float(floor),fraction*max(0.,float(r.max())))
    positive = r>0
    starts = np.flatnonzero(positive & ~np.r_[False,positive[:-1]])
    ends = np.flatnonzero(positive & ~np.r_[positive[1:],False])
    candidates = []
    qualifying_count = 0
    for start,end in zip(starts,ends):
        qualifying = sustained_runs(t[start:end+1],r[start:end+1]>=threshold,sustain)
        if not qualifying:
            continue
        qualifying_count += 1
        if contact-t[end] <= contact_gap+1e-10:
            a,b = qualifying[0]
            candidates.append((int(start),int(end),int(start+a),int(start+b)))
    result = {'state':'not_detected','time':np.nan,'threshold_dps':threshold,
              'episode_start_s':np.nan,'episode_end_s':np.nan,'episode_end_gap_ms':np.nan,
              'threshold_run_duration_ms':np.nan,'qualifying_positive_episodes':qualifying_count,
              'near_contact_qualifying_episodes':len(candidates),'episode_left_censored':False}
    if not candidates:
        return result
    start,end,a,b = candidates[-1]
    censored = a==0
    result.update(state='left_censored' if censored else 'detected',
                  time=np.nan if censored else float(t[a]),
                  episode_start_s=float(t[start]),episode_end_s=float(t[end]),
                  episode_end_gap_ms=1000*(contact-t[end]),
                  threshold_run_duration_ms=1000*(t[b]-t[a]),episode_left_censored=bool(start==0))
    return result

extract_first = extract

def extract(g,lookback=LOOKBACK_S,cutoff=FILTER_HZ,floor=RATE_FLOOR_DPS,
            fraction=PEAK_FRACTION,sustain=SUSTAIN_S,contact_gap=FINAL_EPISODE_CONTACT_GAP_S):
    row = extract_first(g,lookback,cutoff,floor,fraction,sustain)
    tr = prepared_trace(g,lookback,cutoff)
    t = tr['time'][tr['mask']]
    for segment in SEGMENTS:
        final = detect_final_episode(t,tr[segment+'_rate'][tr['mask']],tr['hi'],
                                     floor,fraction,sustain,contact_gap)
        row.update({segment+'_final_onset_state':final['state'],
                    segment+'_final_onset_s':final['time'],
                    segment+'_final_onset_ms_from_plant':1000*(final['time']-row['fp_100_time']),
                    segment+'_final_threshold_dps':final['threshold_dps'],
                    segment+'_final_episode_start_ms_from_plant':1000*(final['episode_start_s']-row['fp_100_time']),
                    segment+'_final_episode_end_ms_from_plant':1000*(final['episode_end_s']-row['fp_100_time']),
                    segment+'_final_episode_end_gap_ms':final['episode_end_gap_ms'],
                    segment+'_final_threshold_run_duration_ms':final['threshold_run_duration_ms'],
                    segment+'_qualifying_positive_episodes':final['qualifying_positive_episodes'],
                    segment+'_near_contact_qualifying_episodes':final['near_contact_qualifying_episodes'],
                    segment+'_final_episode_left_censored':final['episode_left_censored']})
    valid = all(row[s+'_final_onset_state']=='detected' for s in SEGMENTS)
    lag = 1000*(row['torso_final_onset_s']-row['pelvis_final_onset_s']) if valid else np.nan
    row['final_onset_pair_valid'] = valid
    row['final_uprighting_offset_ms'] = lag
    row['final_onset_order'] = ('unavailable' if not valid else
        'unresolved_within_one_sample' if abs(lag)<=row['sample_interval_ms']+1e-7 else
        'pelvis_first' if lag>0 else 'torso_first')
    return row

def measurement_view(data,definition):
    view = data.copy()
    if definition=='first_sustained':
        return view
    if definition!='final_near_contact':
        raise ValueError('unknown measurement definition')
    for standard,final in [('uprighting_offset_ms','final_uprighting_offset_ms'),
                           ('onset_pair_valid','final_onset_pair_valid'),
                           ('onset_order','final_onset_order')]:
        view[standard] = view[final]
    for s in SEGMENTS:
        view[s+'_onset_state'] = view[s+'_final_onset_state']
    return view

In [ ]:
features, audit = build_table()
audit.to_csv(OUT / 'exclusion_audit.csv', index=False)
display(audit.status.value_counts().rename('swings').to_frame())
if features.empty:
    raise ValueError('No swings passed signal/event checks; inspect exclusion_audit.csv.')
features = features.merge(poi[['session_swing'] + list(OUTCOMES.values())],
                          on='session_swing', how='left', validate='one_to_one')
features = features.rename(columns={v:k for k,v in OUTCOMES.items()})
for column in OUTCOMES:
    features[column] = pd.to_numeric(features[column], errors='coerce').replace([np.inf,-np.inf], np.nan)
features['event_duration_reference_eligible'] = features.plant_to_contact_ms.between(*REFERENCE_PLANT_CONTACT_RANGE_MS)
features['large_lag_inspection_flag'] = features.uprighting_offset_ms.abs()>LAG_INSPECTION_THRESHOLD_MS
features['duration_inspection_flag'] = ~features.event_duration_reference_eligible
features['loading_to_plant_inspection_flag'] = features.loading_to_plant_ms>LOADING_TO_PLANT_INSPECTION_MAX_MS
features['combined_duration_reference_eligible'] = features.event_duration_reference_eligible & ~features.loading_to_plant_inspection_flag
features['inspection_flag'] = features.large_lag_inspection_flag | features.duration_inspection_flag | features.loading_to_plant_inspection_flag
features.to_csv(OUT / 'swing_metrics.csv', index=False)
state_counts = pd.concat([features[s+'_onset_state'].value_counts().rename(s)
                         for s in SEGMENTS], axis=1).fillna(0).astype(int)
state_counts.to_csv(OUT / 'onset_state_counts.csv')
display(state_counts)
display(features.onset_order.value_counts().rename('swings').to_frame())
paired = features.loc[features.onset_pair_valid].copy()
print(f'{len(features)} retained swings; {len(paired)} with two detected onsets.')
print('Offsets in samples refer to the CSV sampling grid, not camera-video frames.')
if not paired.empty:
    display(paired[['uprighting_offset_ms', 'torso_minus_pelvis_peak_rate_dps']]
            .describe(percentiles=[.1,.25,.5,.75,.9]))

In [ ]:
def session_profile(g):
    valid = g.loc[g.onset_pair_valid]
    n = len(valid)
    x = valid.uprighting_offset_ms
    enough = n >= MIN_VALID_SWINGS
    return {'retained_swings': len(g), 'valid_onset_pairs': n,
            'pair_detection_fraction': n/len(g), 'consistency_eligible': enough,
            'lag_median_ms': x.median() if n else np.nan,
            'lag_sd_ms': x.std(ddof=1) if enough else np.nan,
            'lag_iqr_ms': x.quantile(.75)-x.quantile(.25) if enough else np.nan,
            'pelvis_first_fraction': valid.onset_order.eq('pelvis_first').mean() if n else np.nan,
            'torso_first_fraction': valid.onset_order.eq('torso_first').mean() if n else np.nan,
            'unresolved_fraction': valid.onset_order.eq('unresolved_within_one_sample').mean() if n else np.nan,
            **{s+'_censored_swings': int(g[s+'_onset_state'].eq('left_censored').sum()) for s in SEGMENTS}}

session_rows = []
for (hitter, session), g in features.groupby(['hitter_id','session_id'], sort=False):
    session_rows.append({'hitter_id': hitter, 'session_id': session, **session_profile(g)})
sessions = pd.DataFrame(session_rows)
sessions.to_csv(OUT / 'session_repeatability.csv', index=False)
hitter_summary = features.groupby('hitter_id').agg(
    retained_swings=('session_swing','size'), valid_onset_pairs=('onset_pair_valid','sum'),
    lag_median_ms=('uprighting_offset_ms','median'))
hitter_summary['pair_detection_fraction'] = hitter_summary.valid_onset_pairs/hitter_summary.retained_swings
hitter_summary.to_csv(OUT / 'hitter_summary.csv')
display(sessions.head(12))

def bootstrap_median(values, n=BOOTSTRAPS, seed=SEED):
    x = np.asarray(values, float)
    x = x[np.isfinite(x)]
    if len(x) < 5:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    estimates = [np.median(rng.choice(x, size=len(x), replace=True)) for _ in range(n)]
    return tuple(np.quantile(estimates, [.025,.975]))

hm = hitter_summary.lag_median_ms.dropna()
ci = bootstrap_median(hm)
cohort = pd.DataFrame([{'retained_swings':len(features), 'detected_pairs':len(paired),
    'retained_hitters':features.hitter_id.nunique(), 'hitters_with_lag':len(hm),
    'hitter_weighted_median_lag_ms':hm.median(), 'bootstrap_ci_low_ms':ci[0],
    'bootstrap_ci_high_ms':ci[1]}])
cohort.to_csv(OUT / 'cohort_summary.csv', index=False)
display(cohort)

fig, axes = plt.subplots(1, 3, figsize=(16,4))
if not paired.empty:
    axes[0].hist(paired.uprighting_offset_ms, bins=25, color=COLORS['pelvis'], alpha=.8)
axes[0].axvline(0, color='black', ls='--')
axes[0].set(xlabel='Torso onset − pelvis onset (ms)', ylabel='Swings', title='Detected onset pairs')
if not paired.empty:
    central_lo, central_hi = paired.uprighting_offset_ms.quantile([.01,.99])
    central = paired.uprighting_offset_ms.between(central_lo,central_hi)
    axes[1].hist(paired.loc[central,'uprighting_offset_ms'],bins=25,color=COLORS['pelvis'],alpha=.8)
    axes[1].axvline(0,color='black',ls='--')
    axes[1].set(xlabel='Torso onset − pelvis onset (ms)',ylabel='Swings',
                title=f'Central 98% view: {int((~central).sum())} tails outside view')
eligible = sessions.loc[sessions.consistency_eligible]
axes[2].scatter(eligible.lag_median_ms, eligible.lag_iqr_ms, color=COLORS['torso'], alpha=.7)
axes[2].set(xlabel='Session median lag (ms)', ylabel='Session lag IQR (ms)',
            title=f'Consistency: ≥{MIN_VALID_SWINGS} detected pairs/session')
fig.tight_layout()
fig.savefig(FIG / 'lag_and_consistency.png', dpi=150)
plt.show()

In [ ]:
audit_columns = ['session_swing','hitter_id','session_id','uprighting_offset_ms',
                 'pelvis_onset_ms_from_plant','torso_onset_ms_from_plant',
                 'plant_to_contact_ms','loading_to_plant_ms','search_duration_ms',
                 'pelvis_threshold_dps','torso_threshold_dps','onset_order',
                 'duration_inspection_flag','loading_to_plant_inspection_flag','large_lag_inspection_flag',
                 'final_uprighting_offset_ms','pelvis_final_onset_ms_from_plant','torso_final_onset_ms_from_plant',
                 'pelvis_final_onset_state','torso_final_onset_state']
inspection = features.loc[features.inspection_flag,audit_columns].copy()
inspection['absolute_lag_ms'] = inspection.uprighting_offset_ms.abs()
inspection = inspection.sort_values('absolute_lag_ms',ascending=False,na_position='last')
inspection.to_csv(OUT/'trial_inspection_audit.csv',index=False)
print('Flagged trials are retained in the full primary analysis.')
print('Reference plant-to-contact interval (ms):',REFERENCE_PLANT_CONTACT_RANGE_MS)
print('Loading-to-plant duration inspection maximum (ms):',LOADING_TO_PLANT_INSPECTION_MAX_MS)
print('Large absolute lag inspection threshold (ms):',LAG_INSPECTION_THRESHOLD_MS)
display(inspection.head(20))
reference = features.loc[features.event_duration_reference_eligible].copy()
reference_pairs = reference.loc[reference.onset_pair_valid].copy()

def cohort_profile(label,data):
    valid = data.loc[data.onset_pair_valid]
    hitter_medians = valid.groupby('hitter_id').uprighting_offset_ms.median()
    ci_low,ci_high = bootstrap_median(hitter_medians)
    session_counts = valid.groupby(['hitter_id','session_id']).size()
    return {'cohort':label,'retained_swings':len(data),'detected_pairs':len(valid),
            'hitters_with_lag':len(hitter_medians),'median_swing_lag_ms':valid.uprighting_offset_ms.median(),
            'median_hitter_lag_ms':hitter_medians.median(),'hitter_ci_low_ms':ci_low,'hitter_ci_high_ms':ci_high,
            'pelvis_first_pct':100*valid.onset_order.eq('pelvis_first').mean(),
            'torso_first_pct':100*valid.onset_order.eq('torso_first').mean(),
            'unresolved_pct':100*valid.onset_order.eq('unresolved_within_one_sample').mean(),
            'lag_min_ms':valid.uprighting_offset_ms.min(),'lag_max_ms':valid.uprighting_offset_ms.max(),
            'repeatability_sessions':int((session_counts>=MIN_VALID_SWINGS).sum())}

duration_comparison = pd.DataFrame([cohort_profile('all_retained',features),
                                   cohort_profile('plant_contact_80_200ms',reference)])
duration_comparison.to_csv(OUT/'event_duration_cohort_comparison.csv',index=False)
display(duration_comparison)
reference_sessions = []
for (h,s),g in reference.groupby(['hitter_id','session_id'],sort=False):
    reference_sessions.append({'hitter_id':h,'session_id':s,**session_profile(g)})
pd.DataFrame(reference_sessions).to_csv(OUT/'reference_session_repeatability.csv',index=False)

print('Inspect the plotted event annotations before treating a flagged lag as a biomechanics finding.')
extreme_keys = inspection.dropna(subset=['uprighting_offset_ms']).session_swing.head(MAX_EXTREME_TRIAL_PLOTS)
for key in extreme_keys:
    row = features.loc[features.session_swing.eq(key)].iloc[0]
    tr = prepared_trace(groups[key])
    m = tr['mask']; x = 1000*(tr['time'][m]-row.fp_100_time)
    fig,ax = plt.subplots(2,1,figsize=(12,7),sharex=True)
    for segment in SEGMENTS:
        a = tr[segment+'_angle'][m]-np.interp(tr['lo'],tr['time'],tr[segment+'_angle'])
        ax[0].plot(x,a,color=COLORS[segment],lw=2,label=segment.title())
        ax[1].plot(x,tr[segment+'_rate'][m],color=COLORS[segment],lw=2,label=segment.title())
        ax[1].axhline(row[segment+'_threshold_dps'],color=COLORS[segment],ls=':',alpha=.6)
        if row[segment+'_onset_state']=='detected':
            for panel in ax:
                panel.axvline(row[segment+'_onset_ms_from_plant'],color=COLORS[segment],ls='--')
        if row[segment+'_final_onset_state']=='detected':
            for panel in ax:
                panel.axvline(row[segment+'_final_onset_ms_from_plant'],color=COLORS[segment],ls='-.',
                             label=segment.title()+' final onset')
    for panel in ax:
        for name,label in zip(EVENTS,['10% loading','Full plant','Contact']):
            event_x = 1000*(row[name]-row.fp_100_time)
            panel.axvline(event_x,color='#555555',lw=1,alpha=.6)
            if panel is ax[0]:
                panel.text(event_x,.98,label,transform=panel.get_xaxis_transform(),rotation=90,va='top',fontsize=8)
        panel.legend(loc='best')
    ax[0].set_ylabel('Change in OBP X angle (deg)')
    ax[1].set(xlabel='Time from full foot plant (ms)',ylabel='Posture rate (deg/s)')
    fig.suptitle(f'{key} | lag {row.uprighting_offset_ms:+.1f} ms | final lag {row.final_uprighting_offset_ms:+.1f} ms | loading to plant {row.loading_to_plant_ms:.1f} ms')
    fig.tight_layout()
    plot_number = int(list(extreme_keys).index(key))+1
    fig.savefig(FIG/f'inspection_trial_{plot_number:02d}.png',dpi=150)
    plt.show()

In [ ]:
combined_reference = features.loc[features.combined_duration_reference_eligible].copy()
definition_rows,repeatability_rows = [],[]
for cohort_label,data in [('all_retained',features),('plant_contact_80_200ms',reference),
                          ('plant_contact_and_loading_reference',combined_reference)]:
    for definition in ['first_sustained','final_near_contact']:
        view = measurement_view(data,definition)
        definition_rows.append({'definition':definition,**cohort_profile(cohort_label,view)})
        for (h,s),g in view.groupby(['hitter_id','session_id'],sort=False):
            repeatability_rows.append({'definition':definition,'cohort':cohort_label,
                                       'hitter_id':h,'session_id':s,**session_profile(g)})
definition_summary = pd.DataFrame(definition_rows)
definition_summary.to_csv(OUT/'onset_definition_cohort_comparison.csv',index=False)
pd.DataFrame(repeatability_rows).to_csv(OUT/'onset_definition_repeatability.csv',index=False)
display(definition_summary)
loading_audit = features[['session_swing','hitter_id','loading_to_plant_ms','plant_to_contact_ms',
                         'loading_to_plant_inspection_flag','uprighting_offset_ms','final_uprighting_offset_ms']]
loading_audit.sort_values('loading_to_plant_ms',ascending=False).to_csv(OUT/'loading_to_plant_audit.csv',index=False)
display(loading_audit.sort_values('loading_to_plant_ms',ascending=False).head(15))
definition_comparison = features[['session_swing','hitter_id','session_id','uprighting_offset_ms',
    'final_uprighting_offset_ms','onset_order','final_onset_order','onset_pair_valid','final_onset_pair_valid',
    'loading_to_plant_ms','loading_to_plant_inspection_flag']].copy()
definition_comparison['final_minus_first_lag_ms'] = definition_comparison.final_uprighting_offset_ms-definition_comparison.uprighting_offset_ms
both = definition_comparison.onset_pair_valid & definition_comparison.final_onset_pair_valid
definition_comparison.to_csv(OUT/'swing_onset_definition_comparison.csv',index=False)
transition = pd.crosstab(definition_comparison.onset_order,definition_comparison.final_onset_order)
transition.to_csv(OUT/'onset_definition_order_transitions.csv')
display(transition)
print('Pairs detected under both definitions:',int(both.sum()))
print('Median absolute lag difference (ms):',definition_comparison.loc[both,'final_minus_first_lag_ms'].abs().median())
display(definition_comparison.loc[both].assign(absolute_difference=lambda d:d.final_minus_first_lag_ms.abs())
        .sort_values('absolute_difference',ascending=False).head(15))
state_rows=[]
for s in SEGMENTS:
    for state,count in features[s+'_final_onset_state'].value_counts().items():
        state_rows.append({'segment':s,'final_onset_state':state,'swings':int(count)})
final_states=pd.DataFrame(state_rows)
final_states.to_csv(OUT/'final_episode_state_counts.csv',index=False)
display(final_states)
fig,ax=plt.subplots(1,2,figsize=(12,4))
for definition,color in [('first_sustained',COLORS['pelvis']),('final_near_contact',COLORS['torso'])]:
    view=measurement_view(features,definition)
    x=view.loc[view.onset_pair_valid,'uprighting_offset_ms']
    if not x.empty:
        ax[0].hist(x,bins=30,color=color,alpha=.4,label=definition)
ax[0].axvline(0,color='black',ls=':')
ax[0].set(xlabel='Torso onset − pelvis onset (ms)',ylabel='Swings',title='Two operational onset definitions')
ax[0].legend()
ax[1].scatter(definition_comparison.loc[both,'uprighting_offset_ms'],
              definition_comparison.loc[both,'final_uprighting_offset_ms'],alpha=.5,s=20,color=COLORS['torso'])
ax[1].set(xlabel='First-sustained lag (ms)',ylabel='Final-episode lag (ms)',title='Identical detected trial pairs')
fig.tight_layout()
fig.savefig(FIG/'onset_definition_comparison.png',dpi=150)
plt.show()

In [ ]:
if SELECTED_SWING is None:
    selected = (paired.loc[(paired.uprighting_offset_ms-paired.uprighting_offset_ms.median()).abs().idxmin()]
                if not paired.empty else features.iloc[0])
else:
    candidates = features.loc[features.session_swing.eq(str(SELECTED_SWING))]
    if candidates.empty:
        raise ValueError('SELECTED_SWING was not retained; inspect the audit or choose another ID.')
    selected = candidates.iloc[0]
selected_key = str(selected.session_swing)
trace = prepared_trace(groups[selected_key])
mask = trace['mask']
time_ms = 1000*(trace['time'][mask]-selected.fp_100_time)

def draw_selected(ax_angle, ax_rate):
    for s in SEGMENTS:
        angle = trace[s+'_angle'][mask]-np.interp(trace['lo'], trace['time'], trace[s+'_angle'])
        ax_angle.plot(time_ms, angle, color=COLORS[s], lw=2, label=s.title())
        ax_rate.plot(time_ms, trace[s+'_rate'][mask], color=COLORS[s], lw=2, label=s.title())
        ax_rate.axhline(selected[s+'_threshold_dps'], color=COLORS[s], ls=':', alpha=.5)
        if selected[s+'_onset_state']=='detected':
            for ax in (ax_angle,ax_rate):
                ax.axvline(selected[s+'_onset_ms_from_plant'], color=COLORS[s], ls='--', alpha=.9)
        if selected[s+'_final_onset_state']=='detected':
            for ax in (ax_angle,ax_rate):
                ax.axvline(selected[s+'_final_onset_ms_from_plant'],color=COLORS[s],ls='-.',
                           label=s.title()+' final onset')
    for ax in (ax_angle, ax_rate):
        for name, label in zip(EVENTS, ['10% loading','Foot plant','Contact']):
            x = 1000*(selected[name]-selected.fp_100_time)
            ax.axvline(x, color='#555555', lw=.9, alpha=.55)
            if ax is ax_angle:
                ax.text(x, .98, label, transform=ax.get_xaxis_transform(), rotation=90, va='top', fontsize=8)
        if selected.onset_pair_valid:
            a,b = sorted([selected.pelvis_onset_ms_from_plant, selected.torso_onset_ms_from_plant])
            ax.axvspan(a,b,color='#D5B45C',alpha=.18)
        ax.set_xlabel('Time from full foot plant (ms)')
        ax.legend(loc='lower right')
    ax_angle.set_ylabel('Change in OBP X angle (deg)')
    ax_rate.set_ylabel('Positive-direction posture rate (deg/s)')

fig, ax = plt.subplots(2,1,figsize=(12,8),sharex=True)
draw_selected(*ax)
fig.suptitle(f'{selected_key} | {selected.onset_order} | lag {selected.uprighting_offset_ms:.1f} ms')
fig.tight_layout()
fig.savefig(FIG / 'selected_swing.png', dpi=150)
plt.show()
display(selected.to_frame('value'))

In [ ]:
configs = [('primary', {}), ('lookback_100ms', {'lookback':.100}),
           ('lookback_200ms', {'lookback':.200}), ('floor_10', {'floor':10.}),
           ('floor_30', {'floor':30.}), ('fraction_10pct', {'fraction':.10}),
           ('fraction_20pct', {'fraction':.20}), ('sustain_5ms', {'sustain':.005}),
           ('sustain_20ms', {'sustain':.020}), ('filter_8hz', {'cutoff':8.}),
           ('filter_12hz', {'cutoff':12.}), ('filter_16hz', {'cutoff':16.}),
           ('absolute_only_20dps',{'floor':20.,'fraction':0.}),
           ('absolute_only_40dps',{'floor':40.,'fraction':0.}),
           ('absolute_only_80dps',{'floor':80.,'fraction':0.})]
sens_rows, sens_tables, sens_audits, transition_rows = [], [], [], []
configuration_tables = {}
primary_columns = ['session_swing','uprighting_offset_ms','onset_order',
                   'pelvis_onset_ms_from_plant','torso_onset_ms_from_plant']
reference_ids = set(reference.session_swing)
for label,config in configs:
    table,q = (features.copy(),audit.copy()) if label=='primary' else build_table(**config)
    q['configuration'] = label
    sens_audits.append(q)
    configuration_tables[label] = table
    if table.empty:
        for cohort_label in ['all_retained','plant_contact_80_200ms']:
            sens_rows.append({'configuration':label,'cohort':cohort_label,'retained_swings':0,'detected_pairs':0})
        continue
    long_table = table[['session_swing','uprighting_offset_ms','onset_order',
                       'pelvis_onset_state','torso_onset_state',
                       'pelvis_onset_ms_from_plant','torso_onset_ms_from_plant']].copy()
    long_table['configuration'] = label
    long_table['event_duration_reference_eligible'] = long_table.session_swing.isin(reference_ids)
    sens_tables.append(long_table)
    for cohort_label,base,alternate in [('all_retained',features,table),
            ('plant_contact_80_200ms',reference,table.loc[table.session_swing.isin(reference_ids)])]:
        common = base[primary_columns].merge(alternate[primary_columns],on='session_swing',
                     suffixes=('_primary','_alternate'),validate='one_to_one').dropna(
                     subset=['uprighting_offset_ms_primary','uprighting_offset_ms_alternate'])
        x = common.uprighting_offset_ms_primary
        y = common.uprighting_offset_ms_alternate
        changes = (y-x).abs()
        valid = alternate.loc[alternate.onset_pair_valid]
        primary_definite = common.onset_order_primary.isin(['pelvis_first','torso_first'])
        alternate_definite = common.onset_order_alternate.isin(['pelvis_first','torso_first'])
        definite_both = primary_definite & alternate_definite
        sign_flip = definite_both & common.onset_order_primary.ne(common.onset_order_alternate)
        hm_alt = valid.groupby('hitter_id').uprighting_offset_ms.median()
        sens_rows.append({'configuration':label,'cohort':cohort_label,
            'retained_swings':len(alternate),'detected_pairs':len(valid),
            'median_lag_ms':valid.uprighting_offset_ms.median(),'median_hitter_lag_ms':hm_alt.median(),
            'paired_comparison_swings':len(common),'median_absolute_lag_change_ms':changes.median(),
            'p95_absolute_lag_change_ms':changes.quantile(.95),'max_absolute_lag_change_ms':changes.max(),
            'pelvis_onset_median_absolute_change_ms':(common.pelvis_onset_ms_from_plant_alternate-common.pelvis_onset_ms_from_plant_primary).abs().median(),
            'torso_onset_median_absolute_change_ms':(common.torso_onset_ms_from_plant_alternate-common.torso_onset_ms_from_plant_primary).abs().median(),
            'order_agreement_pct':100*common.onset_order_primary.eq(common.onset_order_alternate).mean(),
            'definite_pairs_both_settings':int(definite_both.sum()),'definite_order_reversals':int(sign_flip.sum()),
            'definite_order_reversal_pct':100*sign_flip.sum()/definite_both.sum() if definite_both.any() else np.nan,
            'pelvis_first_pct':100*valid.onset_order.eq('pelvis_first').mean(),
            'torso_first_pct':100*valid.onset_order.eq('torso_first').mean(),
            'unresolved_pct':100*valid.onset_order.eq('unresolved_within_one_sample').mean(),
            'lag_rank_rho':spearmanr(x,y).statistic if len(common)>2 and x.nunique()>1 and y.nunique()>1 else np.nan,
            **{s+'_censored_swings':int(alternate[s+'_onset_state'].eq('left_censored').sum()) for s in SEGMENTS}})
        transitions = common.groupby(['onset_order_primary','onset_order_alternate']).size()
        for (before,after),count in transitions.items():
            transition_rows.append({'configuration':label,'cohort':cohort_label,
                                    'primary_order':before,'alternate_order':after,'swings':int(count)})
sensitivity = pd.DataFrame(sens_rows)
sensitivity.to_csv(OUT/'sensitivity_summary.csv',index=False)
pd.concat(sens_tables,ignore_index=True).to_csv(OUT/'sensitivity_swing_onsets.csv',index=False)
pd.concat(sens_audits,ignore_index=True).to_csv(OUT/'sensitivity_exclusion_audit.csv',index=False)
pd.DataFrame(transition_rows).to_csv(OUT/'sensitivity_order_transitions.csv',index=False)
print('Per-setting medians can involve different swings. Paired changes use identical trials.')
display(sensitivity[['configuration','cohort','detected_pairs','median_lag_ms',
                     'median_absolute_lag_change_ms','p95_absolute_lag_change_ms',
                     'order_agreement_pct','definite_order_reversals']])

common_ids = set(paired.session_swing)
for table in configuration_tables.values():
    common_ids &= set(table.loc[table.onset_pair_valid,'session_swing']) if not table.empty else set()
fixed_rows = []
for label,table in configuration_tables.items():
    if table.empty:
        continue
    for cohort_label,ids in [('all_common_detected',common_ids),
                             ('reference_common_detected',common_ids & reference_ids)]:
        sub = table.loc[table.session_swing.isin(ids)]
        fixed_rows.append({'configuration':label,**cohort_profile(cohort_label,sub)})
fixed_cohort_summary = pd.DataFrame(fixed_rows)
fixed_cohort_summary.to_csv(OUT/'sensitivity_fixed_cohort_summary.csv',index=False)
print('Fixed-cohort comparisons require detectable pairs under every tested setting.')
print('Full common cohort:',len(common_ids),'reference common cohort:',len(common_ids & reference_ids))
display(fixed_cohort_summary[['configuration','cohort','detected_pairs','median_swing_lag_ms',
                            'median_hitter_lag_ms','pelvis_first_pct','torso_first_pct','unresolved_pct']])

onset_long = pd.concat(sens_tables,ignore_index=True)
stability_rows = []
for key,g in onset_long.groupby('session_swing',sort=False):
    valid = g.dropna(subset=['uprighting_offset_ms'])
    orders = valid.onset_order.value_counts()
    primary_row = features.loc[features.session_swing.eq(key)]
    primary_order = primary_row.onset_order.iloc[0] if not primary_row.empty else 'excluded'
    stability_rows.append({'session_swing':key,'retained_settings':len(g),'detected_pair_settings':len(valid),
        'primary_order':primary_order,'primary_order_agreement_fraction':valid.onset_order.eq(primary_order).mean(),
        'lag_min_ms':valid.uprighting_offset_ms.min(),'lag_max_ms':valid.uprighting_offset_ms.max(),
        'lag_range_ms':valid.uprighting_offset_ms.max()-valid.uprighting_offset_ms.min(),
        'order_categories_observed':len(orders),
        'contains_both_definite_orders':{'pelvis_first','torso_first'}.issubset(set(valid.onset_order)),
        'event_duration_reference_eligible':key in reference_ids})
stability = pd.DataFrame(stability_rows)
stability.to_csv(OUT/'swing_setting_stability.csv',index=False)
display(stability.sort_values('lag_range_ms',ascending=False).head(15))

view = sensitivity.loc[sensitivity.cohort.eq('all_retained')].set_index('configuration').reindex([c[0] for c in configs])
fig,axes = plt.subplots(1,3,figsize=(16,7))
positions = np.arange(len(view))
axes[0].barh(positions,view.order_agreement_pct,color=COLORS['pelvis'])
axes[0].set_yticks(positions,view.index)
axes[0].set(xlim=(0,100),xlabel='Order agreement with primary (%)',title='Same detected trial pairs')
axes[1].barh(positions,view.definite_order_reversal_pct,color=COLORS['torso'])
axes[1].set_yticks(positions,view.index)
axes[1].set(xlim=(0,100),xlabel='Definite order reversals (%)',title='Both settings resolve order')
axes[2].barh(positions,view.p95_absolute_lag_change_ms,color='#B4974E')
axes[2].set_yticks(positions,view.index)
axes[2].set(xlabel='95th percentile absolute lag change (ms)',title='Timing sensitivity')
for ax in axes:
    ax.invert_yaxis()
fig.tight_layout()
fig.savefig(FIG/'setting_sensitivity.png',dpi=150)
plt.show()
minimum_rows=[]
for label,data in [('all_retained',features),('plant_contact_80_200ms',reference)]:
    for minimum in [3,5,7]:
        valid = data.loc[data.onset_pair_valid]
        counts = valid.groupby(['hitter_id','session_id']).session_swing.transform('size')
        session_medians = valid.loc[counts>=minimum].groupby(['hitter_id','session_id']).uprighting_offset_ms.median()
        minimum_rows.append({'cohort':label,'minimum_detected_pairs':minimum,
                            'sessions':len(session_medians),'median_session_lag_ms':session_medians.median()})
pd.DataFrame(minimum_rows).to_csv(OUT/'session_minimum_sensitivity.csv',index=False)
display(pd.DataFrame(minimum_rows))

In [ ]:
final_configs=dict(configuration_tables)
for gap in [.010,.050]:
    label=f'final_contact_gap_{int(gap*1000)}ms'
    table,gap_audit=build_table(contact_gap=gap)
    final_configs[label]=table
    gap_audit.to_csv(OUT/(label+'_exclusion_audit.csv'),index=False)
final_primary=measurement_view(features,'final_near_contact')
final_sensitivity_rows,final_long=[],[]
for label,table in final_configs.items():
    if table.empty:
        continue
    alternate=measurement_view(table,'final_near_contact')
    long=alternate[['session_swing','uprighting_offset_ms','onset_order','onset_pair_valid']].copy()
    long['configuration']=label
    final_long.append(long)
    for cohort_label,ids in [('all_retained',set(features.session_swing)),
                            ('combined_duration_reference',set(combined_reference.session_swing))]:
        base=final_primary.loc[final_primary.session_swing.isin(ids)]
        sub=alternate.loc[alternate.session_swing.isin(ids)]
        common=base[['session_swing','uprighting_offset_ms','onset_order']].merge(
            sub[['session_swing','uprighting_offset_ms','onset_order']],on='session_swing',
            suffixes=('_primary','_alternate'),validate='one_to_one').dropna(
            subset=['uprighting_offset_ms_primary','uprighting_offset_ms_alternate'])
        change=(common.uprighting_offset_ms_alternate-common.uprighting_offset_ms_primary).abs()
        resolved=(common.onset_order_primary.isin(['pelvis_first','torso_first']) &
                  common.onset_order_alternate.isin(['pelvis_first','torso_first']))
        reversals=resolved & common.onset_order_primary.ne(common.onset_order_alternate)
        final_sensitivity_rows.append({'configuration':label,**cohort_profile(cohort_label,sub),
            'paired_comparison_swings':len(common),'median_absolute_lag_change_ms':change.median(),
            'p95_absolute_lag_change_ms':change.quantile(.95),
            'order_agreement_pct':100*common.onset_order_primary.eq(common.onset_order_alternate).mean(),
            'definite_pairs_both':int(resolved.sum()),'definite_order_reversals':int(reversals.sum())})
final_sensitivity=pd.DataFrame(final_sensitivity_rows)
final_sensitivity.to_csv(OUT/'final_episode_sensitivity.csv',index=False)
pd.concat(final_long,ignore_index=True).to_csv(OUT/'final_episode_sensitivity_swings.csv',index=False)
display(final_sensitivity[['configuration','cohort','detected_pairs','median_swing_lag_ms',
                          'paired_comparison_swings','p95_absolute_lag_change_ms',
                          'order_agreement_pct','definite_order_reversals']])
print('Missing final onsets indicate no qualifying episode near contact or left censoring; they are not zero lag.')

In [ ]:
def safe_rho(x,y):
    return float(spearmanr(x,y).statistic) if len(x)>=3 and np.unique(x).size>1 and np.unique(y).size>1 else np.nan

relationship_rows = []
performance_cohorts=[]
for cohort_label,data in [('all_retained',features),('plant_contact_80_200ms',reference),
                          ('plant_contact_and_loading_reference',combined_reference)]:
    for definition in ['first_sustained','final_near_contact']:
        view=measurement_view(data,definition)
        performance_cohorts.append((definition,cohort_label,view.loc[view.onset_pair_valid]))
for definition,cohort_label,pair_data in performance_cohorts:
    for outcome in OUTCOMES:
        complete = pair_data.dropna(subset=[outcome]).copy()
        good = complete.groupby(['hitter_id','session_id']).session_swing.transform('size')>=MIN_VALID_SWINGS
        within = complete.loc[good].copy()
        for column in ['uprighting_offset_ms',outcome]:
            within[column] -= within.groupby(['hitter_id','session_id'])[column].transform('mean')
        counts = complete.groupby('hitter_id').size()
        between = complete.loc[complete.hitter_id.isin(counts[counts>=MIN_VALID_SWINGS].index)]
        between = between.groupby('hitter_id')[['uprighting_offset_ms',outcome]].mean().reset_index()
        for scope, data in [('between_hitter',between),('within_hitter_session',within)]:
            hitter_ids = data.hitter_id.unique()
            estimate = safe_rho(data.uprighting_offset_ms,data[outcome])
            draws = []
            if len(hitter_ids)>=5:
                chunks = [g[['uprighting_offset_ms',outcome]].to_numpy(float) for _,g in data.groupby('hitter_id')]
                rng = np.random.default_rng(SEED)
                for _ in range(BOOTSTRAPS):
                    draw = np.concatenate([chunks[i] for i in rng.integers(0,len(chunks),size=len(chunks))])
                    value = safe_rho(draw[:,0],draw[:,1])
                    if np.isfinite(value):
                        draws.append(value)
            low,high = np.quantile(draws,[.025,.975]) if draws else (np.nan,np.nan)
            relationship_rows.append({'definition':definition,'cohort':cohort_label,'outcome':outcome,'scope':scope,'rows':len(data),
                                      'hitters':len(hitter_ids),'rho':estimate,
                                      'bootstrap_low':low,'bootstrap_high':high})
relationships = pd.DataFrame(relationship_rows)
relationships.to_csv(OUT / 'exploratory_relationships.csv',index=False)
display(relationships)

In [ ]:
if MAKE_ANIMATION:
    try:
        ensure_archive('hitting_landmarks.zip','landmarks.csv')
        nodes = ['left_hip','right_hip','lsjc','rsjc','lejc','rejc',
                 'lwjc','rwjc','lkjc','rkjc','lajc','rajc']
        columns = ['session_swing','time'] + EVENTS + [n+'_'+a for n in nodes for a in 'xyz']
        landmark_chunks = pd.read_csv(RAW / 'landmarks.csv',usecols=columns,
                                      dtype={'session_swing':str},chunksize=100000)
        matching = [c.loc[c.session_swing.eq(selected_key)] for c in landmark_chunks]
        lm = pd.concat(matching,ignore_index=True).sort_values('time').reset_index(drop=True)
        if lm.empty or lm.time.duplicated().any():
            raise ValueError('missing selected landmarks or duplicate time keys')
        le = event_values(lm)
        if any(not np.isclose(le[n],selected[n],atol=.0001,rtol=0) for n in EVENTS):
            raise ValueError('angle/landmark event disagreement')
        lt = lm.time.to_numpy(float)
        use = (lt>=trace['lo']-1e-10)&(lt<=trace['hi']+1e-10)
        display_t = lt[use]
        if len(display_t)!=mask.sum() or not np.allclose(display_t,trace['time'][mask],atol=.0001,rtol=0):
            raise ValueError('angle/landmark time grid mismatch')
        coords = np.stack([lm[[n+'_'+a for a in 'xyz']].to_numpy(float)[use] for n in nodes],axis=1)
        if not np.isfinite(coords).all():
            raise ValueError('nonfinite landmark in displayed interval')
        projected = coords[:,:,[1,2]]
        edges = [('left_hip','right_hip'),('lsjc','rsjc'),('left_hip','lsjc'),('right_hip','rsjc'),
                 ('lsjc','lejc'),('lejc','lwjc'),('rsjc','rejc'),('rejc','rwjc'),
                 ('left_hip','lkjc'),('lkjc','lajc'),('right_hip','rkjc'),('rkjc','rajc')]
        fig = plt.figure(figsize=(13,8))
        grid = fig.add_gridspec(2,2,width_ratios=[1,1.6])
        pose_ax = fig.add_subplot(grid[:,0])
        a_ax = fig.add_subplot(grid[0,1]); r_ax = fig.add_subplot(grid[1,1],sharex=a_ax)
        draw_selected(a_ax,r_ax)
        low = projected.min(axis=(0,1)); high = projected.max(axis=(0,1))
        pose_ax.set(xlim=(low[0]-.15,high[0]+.15),ylim=(low[1]-.15,high[1]+.15),
                    xlabel='Global Y (m)',ylabel='Global Z (m)',title='Reconstructed joint centers')
        pose_ax.set_aspect('equal',adjustable='box')
        lines = [pose_ax.plot([],[],lw=2,color='#455567')[0] for edge in edges]
        dots = pose_ax.scatter([],[],s=20,color='#455567')
        cursors = [ax.axvline(time_ms[0],color='black',lw=1.6) for ax in (a_ax,r_ax)]
        annotation = pose_ax.text(.02,.98,'',transform=pose_ax.transAxes,va='top',fontsize=9)
        frame_indices = np.unique(np.r_[np.arange(0,len(display_t),3),len(display_t)-1])
        def update(frame):
            for line,(n1,n2) in zip(lines,edges):
                p = projected[frame,[nodes.index(n1),nodes.index(n2)]]
                line.set_data(p[:,0],p[:,1])
            dots.set_offsets(projected[frame])
            for cursor in cursors:
                cursor.set_xdata([time_ms[frame],time_ms[frame]])
            annotation.set_text(f'{selected_key}\n{time_ms[frame]:+.1f} ms from plant\n{selected.onset_order}')
            return lines+[dots,annotation]+cursors
        fig.tight_layout()
        anim = FuncAnimation(fig,update,frames=frame_indices,
                             interval=1000*trace['dt']*3,blit=False)
        html = anim.to_jshtml(fps=1/(trace['dt']*3),embed_frames=True)
        plt.close(fig)
        (FIG / 'selected_swing_animation.html').write_text(html,encoding='utf-8')
        display(HTML(html))
        print('Saved selected_swing_animation.html')
    except (OSError,ValueError,KeyError) as exc:
        print('Animation unavailable:',exc)
        print('The angle analysis and its exported results remain available.')

In [ ]:
manifest = {'study':'pelvis_torso_uprighting_offset_v3','dataset':'OpenBiomechanics Dataset v1',
    'python':platform.python_version(), 'numpy':np.__version__, 'pandas':pd.__version__,
    'settings':{'lookback_s':LOOKBACK_S,'rate_floor_dps':RATE_FLOOR_DPS,
                'peak_fraction':PEAK_FRACTION,'sustain_s':SUSTAIN_S,'extra_filter_hz':FILTER_HZ,
                'minimum_valid_swings':MIN_VALID_SWINGS,'reference_plant_contact_range_ms':REFERENCE_PLANT_CONTACT_RANGE_MS,
                'loading_to_plant_inspection_max_ms':LOADING_TO_PLANT_INSPECTION_MAX_MS,
                'final_episode_contact_gap_s':FINAL_EPISODE_CONTACT_GAP_S,
                'lag_inspection_threshold_ms':LAG_INSPECTION_THRESHOLD_MS,'bootstraps':BOOTSTRAPS,'seed':SEED},
    'archive_expected_sha256':ARCHIVE_HASHES,
    'input_sha256':{name:file_hash(RAW/name) for name in
                    ['joint_angles.csv','metadata.csv','poi_metrics.csv','landmarks.csv'] if (RAW/name).exists()},
    'final_episode_definition':'last contiguous positive-rate episode with sustained threshold crossing ending within contact gap; no earlier fallback',
    'first_onset_definition_preserved':True,'primary_retains_duration_flags':True,'sensitivity_settings':dict(configs),
    'reference_swings':len(reference),'reference_detected_pairs':len(reference_pairs),
    'selected_swing':selected_key,'retained_swings':len(features),'detected_pairs':len(paired),
    'lag_definition':'1000 * (torso_detected_onset_s - pelvis_detected_onset_s)',
    'onset_resolution':'sampled rates; unresolved order within one sample interval',
    'results_directory':str(OUT),'figures_directory':str(FIG)}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

print('Detected primary pairs:',len(paired),'/',len(features))
print('Reference-duration pairs:',len(reference_pairs),'/',len(reference))
print('Primary median lag (ms):',paired.uprighting_offset_ms.median())
print('Reference-duration median lag (ms):',reference_pairs.uprighting_offset_ms.median())
print('Flagged trials for inspection:',len(inspection))
print('Results:',OUT)
print('Figures:',FIG)
print('No universal sequence, ideal lag, or causal performance benefit is assumed.')

print('Final-episode detected pairs:',int(features.final_onset_pair_valid.sum()))
print('Final-episode median lag (ms):',features.final_uprighting_offset_ms.median())
print('Loading-to-plant flagged swings:',int(features.loading_to_plant_inspection_flag.sum()))
print('Compare definitions and detection counts; the final-episode definition is not ground truth.')